<a href="https://colab.research.google.com/github/hcristosm/image_batch_upscale/blob/main/batch_upscale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import shutil
import zipfile
from PIL import Image
from google.colab import files
from IPython.display import clear_output

# 1. Verifica se já está instalado para economizar tempo
if not os.path.exists('/content/Real-ESRGAN'):
    print("Instalando pacotes (isso só ocorre se a sessão foi reiniciada)...")
    !git clone https://github.com/xinntao/Real-ESRGAN.git 2> /dev/null
    os.chdir('/content/Real-ESRGAN')
    !pip install -q basicsr facexlib gfpgan
    !pip install -q -r requirements.txt
    !python setup.py develop > /dev/null
    !wget -q -nc https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights
    !sed -i 's/torchvision.transforms.functional_tensor/torchvision.transforms.functional/g' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
else:
    os.chdir('/content/Real-ESRGAN')

clear_output()

# 2. Resetar pastas
input_dir = 'inputs'
output_dir = 'results'
if os.path.exists(input_dir): shutil.rmtree(input_dir)
if os.path.exists(output_dir): shutil.rmtree(output_dir)
os.makedirs(input_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print("📦 Selecione sua(s) foto(s) ou um arquivo .zip...")
uploaded = files.upload()

if uploaded:
    print("\n⚙️ Preparando arquivos...")

    # Extração inteligente (ignora subpastas se for um zip)
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                for member in zip_ref.namelist():
                    if not member.endswith('/'): # Pula diretórios
                        source = zip_ref.open(member)
                        target_name = os.path.basename(member)
                        if target_name:
                            with open(os.path.join(input_dir, target_name), "wb") as target:
                                shutil.copyfileobj(source, target)
            os.remove(filename)
        else:
            shutil.move(filename, os.path.join(input_dir, filename))

    # Forçar RGB
    arquivos = os.listdir(input_dir)
    print(f"📸 Encontrada(s) {len(arquivos)} imagem(ns). Ajustando formato de cor...")
    for file in arquivos:
        img_path = os.path.join(input_dir, file)
        try:
            with Image.open(img_path) as img:
                if img.mode != 'RGB':
                    img.convert('RGB').save(img_path)
        except Exception:
            pass

    print("\n🚀 Iniciando Upscale...")
    print("⚠️ Dica: O parâmetro '--tile 512' foi ativado para evitar falta de memória na placa de vídeo.\n")

    # Processamento com --tile 512 inserido
    !python inference_realesrgan.py -n RealESRGAN_x4plus -i inputs -o results --outscale 4 --face_enhance --tile 512

    # Verificação de segurança
    resultados = os.listdir(output_dir)
    if len(resultados) == 0:
        print("\n❌ ERRO FATAL: O processamento falhou. Nenhuma imagem foi salva na pasta de resultados.")
        print("Verifique os textos impressos acima desta mensagem para ver o erro técnico (possível arquivo corrompido ou formato não suportado).")
    else:
        print(f"\n✅ SUCESSO! {len(resultados)} imagem(ns) processada(s).")
        print("🗜️ Compactando e iniciando download automático...")
        shutil.make_archive('fotos_upscaled', 'zip', 'results')
        files.download('fotos_upscaled.zip')
else:
    print("\n❌ Nenhum arquivo foi enviado. Rode a célula novamente.")

📦 Selecione sua(s) foto(s) ou um arquivo .zip...


Saving Pedido UP BABY JUL:26.zip to Pedido UP BABY JUL:26.zip

⚙️ Preparando arquivos...
📸 Encontrada(s) 63 imagem(ns). Ajustando formato de cor...

🚀 Iniciando Upscale...
⚠️ Dica: O parâmetro '--tile 512' foi ativado para evitar falta de memória na placa de vídeo.

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Testing 0 46269_000101-2c262728a9c960a7832fcddc600a18f0_340x450
	Tile 1/1
Testing 1 46269_112511-29539a6706ad8f46539a120c15428299_340x450
	Tile 1/1
Testing 2 46269_120722-e1e4656acd437e5353f32984a465aca0_340x450
	Tile 1/1
Te

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title teste
import os
import shutil
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

# 1. Instalação do repositório e modelos
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
!pip install basicsr facexlib gfpgan
!pip install -r requirements.txt
!python setup.py develop
!wget https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights

# 2. PATCH: Corrige a incompatibilidade do Torchvision no Python 3.12 / PyTorch recente
!sed -i 's/torchvision.transforms.functional_tensor/torchvision.transforms.functional/g' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py

# Criar pastas de trabalho
os.makedirs('inputs', exist_ok=True)
os.makedirs('results', exist_ok=True)

# 3. Upload de 1 arquivo
print("\nSelecione 1 foto para o teste:")
uploaded = files.upload()

if uploaded:
    file_name = list(uploaded.keys())[0]
    input_path = os.path.join('inputs', file_name)
    shutil.move(file_name, input_path)

    print(f"\nProcessando '{file_name}' com upscale 4x...")

    # Executar upscale na imagem enviada
    !python inference_realesrgan.py -n RealESRGAN_x4plus -i "{input_path}" -o results --outscale 4 --face_enhance

    # Localizar o arquivo gerado
    name_without_ext, ext = os.path.splitext(file_name)
    output_path = os.path.join('results', f'{name_without_ext}_out{ext}')

    # Exibir Antes vs Depois lado a lado
    if os.path.exists(output_path):
        img_orig = Image.open(input_path)
        img_out = Image.open(output_path)

        fig, axes = plt.subplots(1, 2, figsize=(16, 8))

        axes[0].imshow(img_orig)
        axes[0].set_title(f"ORIGINAL\nDimensões: {img_orig.size[0]} x {img_orig.size[1]} px", fontsize=12)
        axes[0].axis('off')

        axes[1].imshow(img_out)
        axes[1].set_title(f"UPSCALED (4x)\nDimensões: {img_out.size[0]} x {img_out.size[1]} px", fontsize=12)
        axes[1].axis('off')

        plt.tight_layout()
        plt.show()

        # Baixar o resultado individual
        files.download(output_path)